# Mini Projeto PauloBarbatoFogacaAlmeida T8

**Aluno:** Paulo Barbato Fogaça de Almeida

**Turma:** T8 Análise de Dados com Python  

**Base:** Varejo (Kaggle: https://www.kaggle.com/datasets/namespaiva/base-varejo/data )

---

**Como funciona:** o notebook roda do início ao fim, imprime os dados.

As decisões, as conclusões estão no `README.md`.

# Importação e diagnóstico

## Importa o data set

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("namespaiva/base-varejo")

print("Path to dataset files:", path)

# Pega os datasets e passa para csv
import glob, os, shutil
csvs = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)

# Copia para a pasta de trabalho
shutil.copy(csvs[0], "varejo.csv")

Using Colab cache for faster access to the 'base-varejo' dataset.
Path to dataset files: /kaggle/input/base-varejo


'varejo.csv'

## Diagnóstico


In [2]:
import pandas as pd

ARQUIVO_ENTRADA = "varejo.csv"
ARQUIVO_SAIDA = "varejo_limpo.csv"

In [3]:
df_original = pd.read_csv(ARQUIVO_ENTRADA, sep=";")
df_original.head()

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,NaN,NaN,NaN,NaN
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,NaN,NaN,NaN,NaN
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,NaN,NaN,NaN,NaN
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,NaN,NaN,NaN,NaN


In [4]:
print("Registros:", df_original.shape[0])
print("Colunas:  ", df_original.shape[1])
print("\nNomes das colunas:")
print(list(df_original.columns))
print("\nTipos de dados:")
print(df_original.dtypes)

Registros: 830000
Colunas:   14

Nomes das colunas:
['DATA', 'CO_ID', 'CL_ID', 'CL_GENERO', 'CL_EC', 'CL_FHL', 'CL_SEG', 'PR_ID', 'PR_CAT', 'PR_NOME', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']

Tipos de dados:
DATA            object
CO_ID            int64
CL_ID            int64
CL_GENERO       object
CL_EC            int64
CL_FHL           int64
CL_SEG          object
PR_ID            int64
PR_CAT          object
PR_NOME         object
Unnamed: 10    float64
Unnamed: 11    float64
Unnamed: 12    float64
Unnamed: 13    float64
dtype: object


In [5]:
# Colunas da base, nomes reais

# colunas de texto
COLUNAS_TEXTO = ["CL_GENERO", "CL_SEG", "PR_CAT", "PR_NOME"]
COL_ID_COMPRA = "CO_ID"
COL_ID_CLIENTE = "CL_ID"
COL_ID_PRODUTO = "PR_ID"
# coluna numérica analisada: característica do cliente
COL_NUM = "CL_FHL"
# 1 groupby: categoria do produto
COL_GRUPO_1 = "PR_CAT"
# 2 groupby: gênero do cliente
COL_GRUPO_2 = "CL_GENERO"

### 1.1 O que cada linha representa

Para compar o número de linhas com o de compras, clientes e produtos distintos.

In [6]:
n_linhas = len(df_original)
n_compras = df_original[COL_ID_COMPRA].nunique()
n_clientes = df_original[COL_ID_CLIENTE].nunique()
n_produtos = df_original[COL_ID_PRODUTO].nunique()

print(f"Linhas (registros):       {n_linhas:,}")
print(f"Compras distintas (CO_ID):  {n_compras:,}")
print(f"Clientes distintos (CL_ID): {n_clientes:,}")
print(f"Produtos distintos (PR_ID): {n_produtos:,}")
print(f"\nMédia de linhas por compra:  {n_linhas / n_compras:.2f}")
print(f"Média de linhas por cliente: {n_linhas / n_clientes:.2f}")

print('\n')
if n_linhas == n_compras:
    print("Há mesma quantidade linhas e compras.")
elif n_linhas > n_compras:
    print("Há mais linhas do que compras.")
else:
    print("Há menos linhas do que compras.")

Linhas (registros):       830,000
Compras distintas (CO_ID):  18,471
Clientes distintos (CL_ID): 1,000
Produtos distintos (PR_ID): 229

Média de linhas por compra:  44.94
Média de linhas por cliente: 830.00


Há mais linhas do que compras.


### 1.2 Significado das colunas

| Coluna | Significado (inferido) | Tipo |
|---|---|---|
| `DATA` | data do registro | texto |
| `CO_ID` | identificador da compra | id (número) |
| `CL_ID` | identificador do cliente | id (número) |
| `CL_GENERO` | gênero do cliente | texto |
| `CL_EC` | código de estado civil do cliente | número (código) |
| `CL_FHL` | número de filhos do cliente | número |
| `CL_SEG` | segmento do cliente | texto |
| `PR_ID` | identificador do produto | id (número) |
| `PR_CAT` | categoria do produto | texto |
| `PR_NOME` | nome do produto | texto |

### 1.3 Qualidade dos dados

Todas as quantidades, inclusive as que forem zero.


In [7]:
# Valores ausentes por coluna
nulos = df_original.isna().sum()
print(nulos)

# Colunas 100% vazias (as "Unnamed")
colunas_vazias = [c for c in df_original.columns if df_original[c].isna().all()]
print("\nColunas 100% vazias:", colunas_vazias)

# Ausentes nas colunas que têm dados
nulos_uteis = int(df_original.drop(columns=colunas_vazias).isna().sum().sum())
print("Ausentes nas colunas com dados:", nulos_uteis)

DATA                0
CO_ID               0
CL_ID               0
CL_GENERO           0
CL_EC               0
CL_FHL              0
CL_SEG              0
PR_ID               0
PR_CAT              0
PR_NOME             0
Unnamed: 10    830000
Unnamed: 11    830000
Unnamed: 12    830000
Unnamed: 13    830000
dtype: int64

Colunas 100% vazias: ['Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']
Ausentes nas colunas com dados: 0


In [8]:
# Linhas duplicadas (idênticas em todas as colunas)
dups_originais = int(df_original.duplicated().sum())
print("Linhas duplicadas completas:", dups_originais)

if dups_originais > 0:
    print("\nExemplos que possuem cópias:")
    display(df_original[df_original.duplicated(keep=False)]
            .sort_values([COL_ID_COMPRA, COL_ID_CLIENTE, COL_ID_PRODUTO]).head(6))

# O mesmo cliente (ou a mesma compra) em várias linhas não é duplicata.
print("\nLinhas cujo CL_ID já apareceu antes:",
      int(df_original[COL_ID_CLIENTE].duplicated().sum()))

Linhas duplicadas completas: 96553

Exemplos que possuem cópias:


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
40,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
7,01/02/2019,1000,534,M,4,1,C,11,ALIMENTOS,AZEITE,NaN,NaN,NaN,NaN
46,01/02/2019,1000,534,M,4,1,C,11,ALIMENTOS,AZEITE,NaN,NaN,NaN,NaN
14,01/02/2019,1000,534,M,4,1,C,13,ALIMENTOS,BANANA,NaN,NaN,NaN,NaN
19,01/02/2019,1000,534,M,4,1,C,13,ALIMENTOS,BANANA,NaN,NaN,NaN,NaN



Linhas cujo CL_ID já apareceu antes: 829000


In [9]:
# Verifica textos vazios ou compostos apenas por espaços
for col in COLUNAS_TEXTO:
    contagem = df_original[col].value_counts()
    so_espacos = contagem[contagem.index.astype(str).str.strip() == ""].sum()
    print(f"{col}: {so_espacos} textos vazios ou só com espaços")

CL_GENERO: 0 textos vazios ou só com espaços
CL_SEG: 0 textos vazios ou só com espaços
PR_CAT: 0 textos vazios ou só com espaços
PR_NOME: 0 textos vazios ou só com espaços


## 2. Limpeza dados

Remove colunas totalmente vazias, padroniza texto e remove duplicatas completas

In [10]:
# Remove colunas sem nenhum valor
def eliminar_colunas_vazias(df):
    return df.dropna(axis=1, how="all")

In [11]:
# tira espaços extras, unifica a caixa e transforma texto vazio em ausente
def padronizar_texto(df, colunas_texto, caixa="lower"):
    df = df.copy()
    for col in colunas_texto:
        s = (df[col].astype("string")
                    .str.strip() # espaços nas pontas
                    .str.replace(r"\s+", " ", regex=True)) # espaços duplos no meio
        s = getattr(s.str, caixa)() # caixa única
        df[col] = s.replace("", pd.NA) # vazio vira ausente
    return df

### 2.1 Remoção das duplicatas

Linhas idênticas em **todas** as colunas (inclusive `CO_ID`, `CL_ID`, `PR_ID` e `DATA`) são tratadas como duplicata.
Compras diferentes do mesmo cliente **não** entram, pois diferem em `CO_ID`e `DATA`.

In [12]:
# Remove linhas idênticas em todas as colunas
def eliminar_duplicatas(df):
    return df.drop_duplicates()

## Tratar Dados

In [13]:
# Trata a base chamando as funções
def tratar_dados(df, colunas_texto, caixa="lower", remover_duplicatas=True, remover_colunas_vazias=True):
    if remover_colunas_vazias:
        df = eliminar_colunas_vazias(df)
    df = padronizar_texto(df, colunas_texto, caixa)
    if remover_duplicatas:
        df = eliminar_duplicatas(df)
    return df

In [14]:
df_limpo = tratar_dados(df_original, COLUNAS_TEXTO, caixa="lower", remover_duplicatas=True)
df_limpo.head()

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME
0,01/02/2019,1000,534,m,4,1,c,67,bebidas,refrigerante guarana
1,01/02/2019,1000,534,m,4,1,c,70,bebidas,refrigerante outros
2,01/02/2019,1000,534,m,4,1,c,178,higiene,lenco umedecido
3,01/02/2019,1000,534,m,4,1,c,4,alimentos,abacaxi
4,01/02/2019,1000,534,m,4,1,c,175,limpeza,limpador multiuso


## Decisão adotada:
**Manter** os ausentes que existirem e **não preencher**.

Já que preencher exigiria inventar valores.
Enquanto que remover linhas
descartaria dados válidos das outras colunas.

No final contagens e médias do pandas ignoram ausentes automaticamente.

Mas as colunas 100% vazias são removidas
porque não têm informação nenhuma.

In [15]:
print("Ausentes por coluna após o tratamento:")
print(df_limpo.isna().sum())

Ausentes por coluna após o tratamento:
DATA         0
CO_ID        0
CL_ID        0
CL_GENERO    0
CL_EC        0
CL_FHL       0
CL_SEG       0
PR_ID        0
PR_CAT       0
PR_NOME      0
dtype: int64


## Funções reutilizáveis

In [16]:
# Resumo por grupo: registros, compras distintas, clientes distintos e itens por compra
def resumir(df, coluna_grupo, coluna_compra, coluna_cliente):
    resumo = df.groupby(coluna_grupo).agg(
        registros=(coluna_compra, "size"),
        compras=(coluna_compra, "nunique"),
        clientes=(coluna_cliente, "nunique"),
    )
    resumo["itens_por_compra"] = (resumo["registros"] / resumo["compras"]).round(2)
    return resumo.sort_values("clientes", ascending=False)

In [17]:
# Dicionário com métricas de qualidade
def diagnostico(df, colunas_texto):
    vazios = 0
    for col in colunas_texto:
        s = df[col].dropna().astype(str)
        vazios += int((s.str.strip() == "").sum())
    return {
        "registros": len(df),
        "colunas": df.shape[1],
        "colunas 100% vazias": sum(df[c].isna().all() for c in df.columns),
        "células ausentes": int(df.isna().sum().sum()),
        "duplicadas completas": int(df.duplicated().sum()),
        "textos vazios/só espaços": vazios,
    }

In [18]:
comparacao = pd.DataFrame({
    "antes": diagnostico(df_original, COLUNAS_TEXTO),
    "depois": diagnostico(df_limpo, COLUNAS_TEXTO),
})
display(comparacao)

,antes,depois
registros,830000,733447
colunas,14,10
colunas 100% vazias,4,0
células ausentes,3320000,0
duplicadas completas,96553,0
textos vazios/só espaços,0,0


In [19]:
# valores únicos por colunas
for col in COLUNAS_TEXTO:
    print(f"{col}: {df_original[col].nunique()} -> {df_limpo[col].nunique()} valores únicos")

CL_GENERO: 2 -> 2 valores únicos
CL_SEG: 3 -> 3 valores únicos
PR_CAT: 7 -> 7 valores únicos
PR_NOME: 118 -> 118 valores únicos


In [20]:
# Tabela de decisões de limpeza (montada com os números reais desta execução)
removidas_dups = len(df_original) - len(df_limpo)
removidas_dups

96553

|index|Problema|Quantidade|Decisão|Justificativa|
|---|---|---|---|---|
|0|Colunas 100% vazias|4|Removidas|Não contêm nenhuma informação|
|1|Valores ausentes nas colunas com dados|0|Nada a tratar|Nenhum valor ausente encontrado|
|2|Linhas duplicadas completas|96553|Removidas|Idênticas em todas as colunas \(inclui ids e data\); compras distintas do mesmo cliente foram preservadas|
|3|Textos vazios ou só com espaços|0|Nada a tratar|Nenhum encontrado|
|4|Variações de caixa/espaços|0|Padronização aplicada|Evita contar a mesma categoria como valores diferentes|
|5|CL\_FHL diferente para o mesmo cliente|0|Nada a tratar|Valor consistente por cliente|

## 3. Estatísticas básicas

Coluna analisada: **`CL_FHL`** (número de filhos, inferido).

É característica do **cliente**, que aparece em várias linhas.
Por isso a **unidade analisada é o cliente**: usamos uma linha por `CL_ID`. Já que se considerar todas as linhas, clientes que compram mais itens pesariam mais na média.

In [21]:
df_clientes = df_limpo.drop_duplicates(subset=COL_ID_CLIENTE)   # 1 linha por cliente
serie = df_clientes[COL_NUM]

estatisticas = pd.Series({
    "contagem de valores válidos": int(serie.count()),
    "média": serie.mean(),
    "mínimo": serie.min(),
    "máximo": serie.max(),
}).round(2)
display(estatisticas)


,0
contagem de valores válidos,1000.00
média,1.14
mínimo,0.00
máximo,4.00


In [22]:
print(f"Unidade analisada: cliente ({len(df_clientes):,} clientes).")
print(f"Média por LINHA (para comparar): {df_limpo[COL_NUM].mean():.2f}  | média por CLIENTE: {serie.mean():.2f}")
print("\nDistribuição de", COL_NUM, "por cliente:")
print(serie.value_counts().sort_index())

Unidade analisada: cliente (1,000 clientes).
Média por LINHA (para comparar): 1.15  | média por CLIENTE: 1.14

Distribuição de CL_FHL por cliente:
CL_FHL
0    527
1    125
2    130
3    121
4     97
Name: count, dtype: int64


In [23]:
media_cli = serie.mean()
pct_com = (serie >= 1).mean() * 100
print(f"Resultado 1: em média, cada cliente tem {media_cli:.2f} em {COL_NUM}.")
print(f"Resultado 2: {pct_com:.1f}% dos clientes têm {COL_NUM} >= 1; o máximo é {serie.max()}"
      + (f", cerca de {serie.max() / media_cli:.1f} vezes a média." if media_cli > 0 else "."))

Resultado 1: em média, cada cliente tem 1.14 em CL_FHL.
Resultado 2: 47.3% dos clientes têm CL_FHL >= 1; o máximo é 4, cerca de 3.5 vezes a média.


## 4. Filtro

Filtro: registros de clientes com `CL_FHL >= 1`, ou seja, clientes com filhos.

**Análise**: o que esse público compra?

In [24]:
df_filtrado = df_limpo[df_limpo[COL_NUM] >= 1]

print(f"Registros (itens) antes do filtro: {len(df_limpo):,}")
print(f"Registros (itens) após o filtro:   {len(df_filtrado):,}  ({len(df_filtrado) / len(df_limpo) * 100:.1f}%)")
print(f"Clientes após o filtro:            {df_filtrado[COL_ID_CLIENTE].nunique():,}")


Registros (itens) antes do filtro: 733,447
Registros (itens) após o filtro:   348,461  (47.5%)
Clientes após o filtro:            473


In [25]:
print("\nCategorias mais frequentes no filtro:")
print(df_filtrado[COL_GRUPO_1].value_counts().head(5))


Categorias mais frequentes no filtro:
PR_CAT
alimentos    182527
higiene       65400
limpeza       61007
bebidas       18273
pet           13593
Name: count, dtype: Int64


In [26]:
# Funções de formatação
fmt = lambda n: f"{int(n):,}".replace(",", ".")          # 830000 -> 830.000
pct = lambda x: f"{x:.1f}".replace(".", ",")             # 12.3 -> 12,3
dec = lambda x, n=2: f"{x:.{n}f}".replace(".", ",")      # 3.45 -> 3,45


## Agrupamentos com `groupby()`

Número de **linhas (itens de compra)** por categoria. Mede volume de itens, não clientes nem faturamento

In [27]:
# Agrupamento 1: registros por categoria de produto
agrup1 = (df_limpo.groupby(COL_GRUPO_1).size().sort_values(ascending=False)
          .rename("registros").to_frame())
agrup1["% do total"] = (agrup1["registros"] / len(df_limpo) * 100).round(2)
agrup1

,registros,% do total
PR_CAT,,
alimentos,384197,52.38
higiene,137702,18.77
limpeza,128632,17.54
bebidas,38264,5.22
pet,28553,3.89
acessorios,12871,1.75
#n/d,3228,0.44


Por gênero, o número de linhas, de **compras distintas** (`CO_ID`) e de **clientes distintos** (`CL_ID`), além de itens por compra.

In [28]:
# Agrupamento 2: registros, compras distintas e clientes distintos por gênero
agrup2 = resumir(df_limpo, COL_GRUPO_2, COL_ID_COMPRA, COL_ID_CLIENTE)
agrup2

,registros,compras,clientes,itens_por_compra
CL_GENERO,,,,
f,382427,9615,519,39.77
m,351020,8856,481,39.64


## 5. Exportação

Salva a base tratada como varejo_limpo.csv

In [29]:
df_limpo.to_csv(ARQUIVO_SAIDA, index=False)
"Salvo arquivo: " + ARQUIVO_SAIDA

'Salvo arquivo: varejo_limpo.csv'

In [30]:

print(f"Registros antes:  {len(df_original):,}")
print(f"Registros depois: {len(df_limpo):,}")
print(f"Removidos:        {len(df_original) - len(df_limpo):,}")
print(f"Colunas antes: {df_original.shape[1]} | depois: {df_limpo.shape[1]}")

# Conferência: arquivo salvo sem coluna de índice
print("\nColunas do arquivo salvo:", list(pd.read_csv(ARQUIVO_SAIDA, nrows=0).columns))

Registros antes:  830,000
Registros depois: 733,447
Removidos:        96,553
Colunas antes: 14 | depois: 10

Colunas do arquivo salvo: ['DATA', 'CO_ID', 'CL_ID', 'CL_GENERO', 'CL_EC', 'CL_FHL', 'CL_SEG', 'PR_ID', 'PR_CAT', 'PR_NOME']


##  6. Conclusões

Após a execução.

1. A limpeza removeu colunas sem nenhuma informação e linhas repetidas em todas as colunas. A célula abaixo mostra o tamanho da base antes e depois.

In [31]:
removidas = len(df_original) - len(df_limpo)
pct_rem = removidas / len(df_original) * 100

print("Registros antes: ", fmt(len(df_original)))
print("Registros depois:", fmt(len(df_limpo)))
print("Colunas antes:   ", df_original.shape[1])
print("Colunas depois:  ", df_limpo.shape[1])
print("Colunas vazias removidas:", len(colunas_vazias), colunas_vazias)
print("Linhas duplicadas removidas:", fmt(removidas), f"({pct(pct_rem)}%)")

Registros antes:  830.000
Registros depois: 733.447
Colunas antes:    14
Colunas depois:   10
Colunas vazias removidas: 4 ['Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']
Linhas duplicadas removidas: 96.553 (11,6%)


2. Cada linha é um item de uma compra, por isso, contar linhas não é o mesmo que contar compras ou clientes.

In [32]:
n_linhas = len(df_limpo)
n_compras = df_limpo[COL_ID_COMPRA].nunique()
n_clientes = df_limpo[COL_ID_CLIENTE].nunique()

print("Linhas (itens):          ", fmt(n_linhas))
print("Compras distintas:       ", fmt(n_compras))
print("Itens por compra (média):", dec(n_linhas / n_compras))
print("Linhas por cliente (média):", dec(n_linhas / n_clientes, 1))

Linhas (itens):           733.447
Compras distintas:        18.471
Itens por compra (média): 39,71
Linhas por cliente (média): 733,4


3. A tabela mostra quantos clientes distintos, compras e itens existem em cada gênero.

In [33]:
display(agrup2)

n_cli_total = df_limpo[COL_ID_CLIENTE].nunique()
top_gen = agrup2.index[0]
cli_top_gen = agrup2["clientes"].iloc[0]
print(f"Maior grupo: '{top_gen}' com {fmt(cli_top_gen)} de {fmt(n_cli_total)} clientes ({pct(cli_top_gen / n_cli_total * 100)}%)")

,registros,compras,clientes,itens_por_compra
CL_GENERO,,,,
f,382427,9615,519,39.77
m,351020,8856,481,39.64


Maior grupo: 'f' com 519 de 1.000 clientes (51,9%)


4. Quantidade de filhos por cliente, `CL_FHL` é uma característica do cliente, a análise usa cada cliente uma única vez

In [34]:
df_clientes = df_limpo.drop_duplicates(subset=COL_ID_CLIENTE)   # 1 linha por cliente
serie = df_clientes[COL_NUM]
pct_com = (serie >= 1).mean() * 100

print("Unidade analisada: cliente |", fmt(len(df_clientes)), "clientes")
print("Contagem válida:", int(serie.count()))
print("Média: ", dec(serie.mean()))
print("Mínimo:", serie.min())
print("Máximo:", serie.max())
print(f"Clientes com {COL_NUM} >= 1: {pct(pct_com)}%")

Unidade analisada: cliente | 1.000 clientes
Contagem válida: 1000
Média:  1,14
Mínimo: 0
Máximo: 4
Clientes com CL_FHL >= 1: 47,3%


### Recomendação

Concentrar o planejamento de estoque e as campanhas na categoria que mais vende e corrigir na origem os registros sem categoria definida.

### Limitações

A base não tem valor em R$ nem quantidade, então não é possível medir faturamento.

Cada linha é um item, e não um cliente.

Os significados de CL_FHL, CL_EC e CL_SEG foram inferidos pelos nomes das colunas, e a DATA não foi convertida.